# 09｜箱形圖 (Box Plot)：三個客服團隊處理時間的五數概括與穩定度評估

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/ai-excel-analytics/blob/main/02_%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E6%B1%BA%E5%AE%9A%E9%81%A9%E5%90%88%E5%9C%96%E8%A1%A8%E7%9A%84%E6%80%9D%E8%80%83%E8%A6%8F%E5%89%87/Colab%E7%AF%84%E4%BE%8B/09_%E7%AE%B1%E5%BD%A2%E5%9C%96.ipynb)

## 🎯 學習目標與商業情境

- **商業情境**：客服部門評估三個團隊（甲組、乙組、丙組）處理案件的時效。若單看「平均處理時數」：甲組 4.2 小時、乙組 6.8 小時、丙組 6.5 小時。主管看報表第一反應是：「乙組跟丙組怎麼比甲組慢了 50% 以上？乙組是不是效率最差？」
- **統計核心問題**：**箱形圖的五數概括（Min, Q1, Median, Q3, Max）、四分位距 (IQR) 與離群值 (Outlier) 如何徹底翻轉主管的片面判斷？**

> 💡 **上課操作指引**：請學生在瀏覽器開啟空白 Colab 筆記本，一邊聽老師講解箱形圖幾何構造，一邊依序複製以下儲存格貼上執行！

In [ ]:
# [儲存格 1] 設定繁體中文字型與專業繪圖風格
%pip install -q mplfonts

import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

# 載入支援繁體中文的開源字型
for font_file in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(font_file)
sns.set_theme(style="whitegrid")
use_font("Noto Sans CJK SC")
plt.rcParams["axes.unicode_minus"] = False
print("✅ 繁體中文圖表環境準備完成")

In [ ]:
# [儲存格 2] 建立三組客服處理時數資料（各 30 筆），並計算五數概括與四分位距
# 模擬情境設定：
# - 甲組 (標竿型)：高度標準化，集中於 3.5~4.5 小時，無離群點
# - 乙組 (特案卡關型)：常規表現全公司最快 (中位數 3.5 小時！)，但有 2 筆卡死跨部門的極端特案 (22h, 25h)
# - 丙組 (缺乏標準型)：無極端點，但時數散落於 2~12 小時，落差極大 (IQR 超過 5 小時)
np.random.seed(871)
team_a = np.round(np.random.normal(loc=4.0, scale=0.7, size=30).clip(2.6, 5.4), 1)

team_b_normal = np.round(np.random.normal(loc=3.5, scale=0.6, size=28).clip(2.0, 4.8), 1)
team_b_outliers = np.array([22.0, 25.0])
team_b = np.concatenate([team_b_normal, team_b_outliers])

team_c = np.round(np.random.uniform(low=2.0, high=12.0, size=30), 1)

df_list = []
for val in team_a: df_list.append({"團隊": "甲組 (標準化標竿)", "處理時數": val})
for val in team_b: df_list.append({"團隊": "乙組 (特案卡關)", "處理時數": val})
for val in team_c: df_list.append({"團隊": "丙組 (落差懸殊)", "處理時數": val})
df = pd.DataFrame(df_list)

# 計算三組的五數概括與統計指標
def calc_five_number(series):
    q1 = series.quantile(0.25)
    med = series.median()
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper_whisker = q3 + 1.5 * iqr
    outliers = series[series > upper_whisker]
    return pd.Series({
        "平均數 (Mean)": f"{series.mean():.1f} 小時",
        "中位數 (Median)": f"{med:.1f} 小時",
        "Q1 (前 25%)": f"{q1:.1f} 小時",
        "Q3 (前 75%)": f"{q3:.1f} 小時",
        "箱高 IQR (中間 50% 跨度)": f"{iqr:.1f} 小時",
        "離群值數量 (>1.5×IQR)": f"{len(outliers)} 筆"
    })

summary_table = df.groupby("團隊")["處理時數"].apply(calc_five_number).unstack()
display(summary_table)


## 🧭 老師講解重點一：如何向學生拆解箱形圖的幾何結構？

請看上方統計表，若只看第一欄「平均數」，主管一定會誤以為「乙組平均 4.8 小時表現落後於甲組 (3.9 小時)」！但請看「中位數」與「IQR」：

### 📌 箱形圖的四大解剖構造：
1. **箱體的中間橫線（中位數 Median）**：不受極端值干擾的典型水準。
   - **乙組中位數 3.5 小時，比甲組 (4.0 小時) 還要快！** 乙組平時常態案件根本超有效率！
2. **箱子的高度（四分位距 IQR = Q3 - Q1）**：
   - 箱子代表 **中間 50% 核心群體的表現跨度**。
   - **箱子越矮（如甲組 IQR 1.0h、乙組 IQR 1.0h）**：代表團隊作業高度標準化，常規案件處理速度整齊劃一。
   - **箱子越高（如丙組 IQR 5.4h）**：代表表現忽快忽慢、因人而異，缺乏標準 SOP。
3. **上下延伸的鬍鬚（Whiskers）**：正常案件的合理波動範圍（以 1.5 × IQR 為邊界）。
4. **孤立的點（離群值 Outliers）**：
   - 超出鬍鬚外的點（如乙組的 22h, 25h）。這是統計學大師 John Tukey 定義的 **「統計異常值」**，代表發生了常態之外的極端黑天鵝事件！


In [ ]:
# [儲存格 3] 繪製多組箱形圖：對比中位數、箱體高度與紅點離群值
fig, ax = plt.subplots(figsize=(10, 6))

# 繪製箱形圖
palette = ["#60a5fa", "#fb923c", "#a78bfa"]
sns.boxplot(data=df, x="團隊", y="處理時數", palette=palette, width=0.45, 
            flierprops=dict(marker="o", markerfacecolor="#dc2626", markersize=8, markeredgecolor="black"), ax=ax)

# 在箱形圖上加繪各組「平均數 (綠色菱形)」，清楚對照平均數如何被拉偏
means = df.groupby("團隊")["處理時數"].mean()
for i, team in enumerate(["甲組 (標準化標竿)", "乙組 (特案卡關)", "丙組 (落差懸殊)"]):
    m_val = means[team]
    med_val = df[df["團隊"] == team]["處理時數"].median()
    ax.scatter(i, m_val, color="#15803d", marker="D", s=70, zorder=5, label="平均數 (Mean)" if i == 0 else "")
    ax.text(i + 0.25, med_val, f"中位數: {med_val:.1f}h", color="#1e293b", fontweight="bold", va="center", fontsize=9.5)
    ax.text(i + 0.25, m_val, f"平均: {m_val:.1f}h", color="#15803d", fontweight="bold", va="center", fontsize=9.5)

# 標註乙組的離群特例案件
ax.annotate("⚠️ 特案卡關！
兩筆跨部門案件高達 22~25h
(嚴重拉偏平均數至 4.8h)", 
            xy=(1, 23.5), xytext=(1.35, 20), 
            arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.8), 
            fontsize=10, fontweight="bold", color="#dc2626", 
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#fef2f2", edgecolor="#fca5a5"))

ax.set_title("箱形圖診斷：乙組常規中位數最快 (3.5h)，但遭離群特案拉偏；丙組箱體過大 (5.4h) 缺乏標準化", fontsize=13, fontweight="bold", pad=16)
ax.set_xlabel("客服團隊", fontsize=11, labelpad=8)
ax.set_ylabel("案件處理時間（小時）", fontsize=11, labelpad=8)
ax.set_ylim(0, 28)
ax.legend(loc="upper right", frameon=True, facecolor="#f8fafc", edgecolor="#cbd5e1")

plt.tight_layout()
plt.show()


## 📊 老師講解重點二：看圖說商業洞察與主管管理決策

### 1. 箱形圖診斷出的三個團隊真相
- **甲組（標竿型）**：箱體緊湊（IQR 僅 1.0h），中位數 4.0h，沒有任何紅點。代表 SOP 落實徹底，常規案件都在 3~5 小時內高品質結案。
- **乙組（被誤解的英雄團隊）**：中位數 3.5h，常規日常速度其實是全公司最快（比甲組 4.0h 快了半小時）！但頭頂跳出兩筆 22h 與 25h 的紅色離群點，把平均數拉高到 4.8h。這說明乙組同仁日常非常優秀，只是被 2 筆跨部門系統卡死或法律糾紛案件嚴重拖累。
  - 💡 **管理盲點**：若只看平均數，主管會誤判乙組表現落後（4.8h vs 3.9h），甚至痛責第一線客服，徹底打擊優秀員工的士氣！
- **丙組（缺乏標準化）**：箱體拉得極長（IQR 達 5.4h），從 2 小時一路散布到接近 12 小時。雖然沒有紅點，但代表團隊內新人老手表現落差巨大，缺乏規範。

### 2. 客服總監的精準決策行動方針
- **對甲組**：保持現狀，邀請甲組主管整理其內部的作業查檢表 (Checklist)，作為全公司標竿。
- **對乙組**：**絕對不要責怪第一線客服！** 主管應主動介入排查那 2 筆卡關 20 小時以上的特案，推動建立「跨部門卡關逾 8 小時自動升級總監協調機制」，替同仁掃除障礙。
- **對丙組**：**全面啟動 SOP 與教育訓練**，推行師徒制結對，消除個人經驗落差，目標是在下一季將丙組的箱體高度（IQR）壓縮到 2 小時以內！

> 💡 **箱形圖一句訣**：看分布抗極端值首選箱形圖；看中位數認清典型、看箱高 (IQR) 評估穩定度、看紅點排查特例事件！
